# Lab 11: NFL Clustering Practice

This dataset includes receiving characteristics for all NFL players in 2018.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [ ]:
import warnings
warnings.filterwarnings(action='ignore')

In [ ]:
from yellowbrick.datasets import load_nfl
# Load nfl dataset dataset
X, y = load_nfl()

# Specify the features to use for clustering
# Only consider the players that had over 20 targets
features = ['Rec', 'Yds', 'TD', 'Fmb', 'Ctch_Rate']
X = X.query('Tgt >= 20')[features]
X

In [ ]:
X.describe()

## K-means clustering

The first step is to determine how many clusters to use. We can use the elbow method to determine this. Are there any preprocessing steps that we need to do first?

In [ ]:
# Implement preprocessing step(s)
from sklearn.preprocessing import StandardScaler

df_scaled = X.copy()

scaler = StandardScaler()
scaler.fit(df_scaled)
df_scaled = pd.DataFrame(scaler.transform(df_scaled), columns=df_scaled.columns)

In [ ]:
# Use the elbow method to determine the number of clusters to use
from sklearn.cluster import KMeans
from yellowbrick.cluster.elbow import kelbow_visualizer
kelbow_visualizer(KMeans(random_state=54), df_scaled, k=(2,10), 
                  metric='distortion',
                 timings=False)

The next step is to implement the clustering method with the number of clusters determined above. 

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=54)
kmeans.fit(df_scaled)

X['clusters'] = kmeans.labels_
X['clusters'] = X['clusters'].astype('category') # makes seaborn use qualitative color palette

In [ ]:
X.dtypes

In [ ]:
# Find the cluster center for each cluster (for each feature)
centers = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=df_scaled.columns)
centers_scaled = pd.DataFrame(kmeans.cluster_centers_, columns=df_scaled.columns)
centers

In [ ]:
centers_scaled

In [ ]:
# Plot the clusters (separated by color) with their cluster centers
fig, ax = plt.subplots(figsize=(4,4))
ax = sns.scatterplot(x='Rec', y='Ctch_Rate', hue='clusters', ax=ax, data=X)

centers.plot.scatter(x='Rec', y='Ctch_Rate', ax=ax, marker='x', s=80, color='black');

In [ ]:
# Plot a pie chart showing the distribution of data samples for each cluster
from collections import Counter
counts = Counter(kmeans.labels_)
print(counts)
fig1, ax1 = plt.subplots()
ax1.pie(counts.values(), labels=[f'Cluster {i}' for i in counts.keys()], autopct='%1.1f%%',
        shadow=True, startangle=90)
ax1.axis('equal');  # Equal aspect ratio ensures that pie is drawn as a circle.
ax1.set_title('Cluster size distribution');

**Note:** Clusters are not balanced.

In [ ]:
# Plot the cluster centers for each feature on the standard scale
f, axes = plt.subplots(kmeans.n_clusters, 1, figsize=(6, 6), sharex=True)

for i, ax in enumerate(axes):
    center = centers_scaled.loc[i, :]
    maxPC = 1.01 * np.max(np.max(np.abs(center)))
    colors = ['C0' if l>0 else 'C1' for l in center]
    ax.axhline(color='#888888')
    center.plot.bar(ax=ax, color=colors)
    ax.set_ylabel(f'Cluster{i}')
    ax.set_ylim(-maxPC, maxPC)
    if i == 0:
        ax.set_title('Cluster centers per feature on standard scale')


Based on the cluster distribution, was this the optimal number of clusters? Let's check the elbow method again using silhouette index and calinski-harabasz index.

In [ ]:
# Implement elbow method with silhouette index metric
kelbow_visualizer(KMeans(random_state=54), df_scaled, k=(2,10), 
                  metric='silhouette',
                 timings=False)

In [ ]:
# Implement elbow method with calinski_harabasz index metric
kelbow_visualizer(KMeans(random_state=54), df_scaled, k=(2,10), 
                  metric='calinski_harabasz',
                 timings=False)